# Day 37 — Task C Quantitative Metrics Scenarios

Notebook này chỉ minh họa synthetic scenarios. Implementation chính nằm trong `ai-core/quantitative/day37/`.
Không dùng notebook để tạo clinical claim hoặc fatigue diagnosis.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'ai-core' / 'quantitative'))
from day37.repeatability import pair_repeatability, within_subject_cv, robust_mad_ratio, bland_altman
from day37.similarity import cosine_similarity, pearson_similarity, normalized_euclidean
from day37.cocontraction import cocontraction_eligibility, cocontraction_metrics


## Scenario 1 — Repeatability tốt và kém


In [ ]:
stable = np.array([1.00, 1.04, 0.98, 1.02])
unstable = np.array([0.60, 1.40, 0.90, 1.60])
print('stable CV:', within_subject_cv(stable, True))
print('unstable CV:', within_subject_cv(unstable, True))
print('stable rMAD:', robust_mad_ratio(stable))
print('unstable rMAD:', robust_mad_ratio(unstable))


## Scenario 2 — Correlation cao nhưng agreement kém


In [ ]:
session_a = np.array([1, 2, 3, 4, 5], dtype=float)
session_b = session_a + 2.0
print('Pearson:', pearson_similarity(session_a, session_b))
print('Bland-Altman:', bland_altman(session_a, session_b))


## Scenario 3 — Similarity shape và magnitude


In [ ]:
x = np.array([1, 2, 3, 4], dtype=float)
same_shape_larger = 2.0 * x
different_shape = np.array([4, 1, 3, 2], dtype=float)
for name, y in [('same_shape_larger', same_shape_larger), ('different_shape', different_shape)]:
    print(name, {
        'cosine': cosine_similarity(x, y),
        'pearson': pearson_similarity(x, y),
        'normalized_euclidean': normalized_euclidean(x, y),
    })


## Scenario 4 — Co-contraction eligibility và synthetic envelopes


In [ ]:
blocked = cocontraction_eligibility({
    'verified_agonist_antagonist_mapping': False,
    'same_side': True,
    'synchronized_time_base': True,
    'compatible_envelope_units': True,
    'active_phase_present': True,
    'normalization_method': 'mvc_percent',
    'quality_status': 'pass',
})
print('public-dataset-like eligibility:', blocked)

t = np.linspace(0, 1, 1000)
agonist = np.maximum(0, np.sin(2*np.pi*t))
antagonist = np.maximum(0, np.sin(2*np.pi*t + 0.8))
metrics = cocontraction_metrics(agonist, antagonist, dt_seconds=t[1]-t[0], agonist_threshold=0.2, antagonist_threshold=0.2)
print(metrics)


## Scenario 5 — Kết luận an toàn

- Repeatability thấp không đồng nghĩa pathology.
- Similarity thấp không đồng nghĩa abnormality.
- Co-contraction index cao không đồng nghĩa diagnosis.
- Các metrics chỉ được gửi sang Context Engine kèm supportability và provenance.
